In [ ]:
!pip install pandas
!pip install numpy 
!pip install matplotlib

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 

#Display all columns
pd.set_option("display.max_columns", None)

In [ ]:
from pathlib import Path
file_path = Path(data_dir/"ShiftsWorked.csv")
df = pd.read_csv(file_path)
df.head()

In [ ]:
# Number of rows columns 
df.shape

# Column names and data types
df.info()

#descriptive statistics 
df.describe(include = "all").T

#preview random records
df.sample(5, random_state = 42)

In [ ]:
#convert date and time variables to date/time data types
print(df[["Date", "Shift Start", "Shift End"]]. head(10))
df["Date"] = pd.to_datetime(
    df["Date"],
    format = "%Y-%m-%d",
    errors = "coerce") #converts to NaT (not a time)

df["Shift Start"] = pd.to_datetime(
    df["Shift Start"], 
    format = "%H:%M",
    errors = "coerce").dt.time

df["Shift End"] = pd.to_datetime(
    df["Shift End"], 
    format = "%H:%M",
    errors = "coerce").dt.time

print(df.dtypes)
print(df[["Date", "Shift Start", "Shift End"]].head())
print(df[["Date", "Shift Start", "Shift End"]].isna().sum())

# Count missing value by column
missing_values = df.isna().sum()

#count duplicated rows 
duplicate_rows = df.duplicated().sum()

print("\nMissing Values: ")
print(missing_values)

print("\nDuplicate rows: ", duplicate_rows)

In [ ]:
#investigating missing values:
df["Shift"].value_counts(dropna = False)

In [ ]:
#investigate missing shift times by shift type
df.groupby("Shift", dropna = False).agg(
    total_records = ("Record ID", "size"),
    missing_start = ("Shift Start", lambda x: x.isna().sum()),
    missing_end = ("Shift End", lambda x: x.isna().sum())
)

In [ ]:
#Records where only one time is missing 
incomplete_shifts = df[
    df["Shift Start"].isna()!= df["Shift End"].isna()
]
print("Incomplete shift records: ", len(incomplete_shifts))

incomplete_shifts.head(10)

In [ ]:
#examine payment information for missing shift times
df.loc[
    df["Shift Start"].isna(),
    ["Shift", "Daily Pay"]
].groupby("Shift", dropna = False)["Daily Pay"].agg(
    ["count", "min", "mean", "max"]
)

In [ ]:
# Identify working shifts with missing times
working_shifts_missing = df.loc[
    (df["Shift"] != "Rest") &
    (
        df["Shift Start"].isna() |
        df["Shift End"].isna()
    )
]

print(
    "Working shifts with missing times: ",
    len(working_shifts_missing)
)

working_shifts_missing.head()

In [ ]:
# Identify inconsistent rest-day records
rest_inconsistencies = df.loc[
    (df["Shift"] == "Rest") &
    (
        df["Shift Start"].notna() |
        df["Shift End"].notna() |
        (df["Daily Pay"] != 0)
    )
]

print(
    "Inconsistent rest-day records:",
    len(rest_inconsistencies)
)

In [ ]:
# combine date and time values for working shifts
working = df["Shift"] != "Rest"

df["Shift_Start_Datetime"] = pd.NaT
df["Shift_End_Datetime"] = pd.NaT

df.loc[working, "Shift_Start_Datetime"] = pd.to_datetime(
    df.loc[working, "Date"].dt.strftime("%Y-%m-%d")
    + " "
    + df.loc[working, "Shift Start"].astype(str)
)

df.loc[working, "Shift_End_Datetime"] = pd.to_datetime(
    df.loc[working, "Date"].dt.strftime("%Y-%m-%d")
    + " "
    + df.loc[working, "Shift End"].astype(str)
)

In [ ]:
# Identify shifts crossing midnight
overnight = (
    working
    & (
        df["Shift_End_Datetime"]
        < df["Shift_Start_Datetime"]
    )
)

print("Overnight shifts:", overnight.sum())

# Add one day to the end datetime
df.loc[overnight, "Shift_End_Datetime"] += pd.Timedelta(days=1)

In [ ]:
#calculate the duration of each shift
df["Hours_Worked"] = (
    df["Shift_End_Datetime"]
    - df["Shift_Start_Datetime"]
).dt.total_seconds() / 3600

# Assign zero hours to rest days
df.loc[df["Shift"] == "Rest", "Hours_Worked"] = 0

# View the results
df[
    [
        "Date",
        "Shift",
        "Shift_Start_Datetime",
        "Shift_End_Datetime",
        "Hours_Worked"
    ]
].head(10)

In [ ]:
# Summary of hours worked by shift category
df.groupby("Shift")["Hours_Worked"].agg(
    ["count", "min", "mean", "median", "max"]
)

In [ ]:
# Flag working shifts with zero, negative,
# or unusually long durations
invalid_durations = df.loc[
    (df["Shift"] != "Rest")
    & (
        (df["Hours_Worked"] <= 0)
        | (df["Hours_Worked"] > 16)
        | df["Hours_Worked"].isna()
    )
]

print("Potentially invalid shift durations:", len(invalid_durations))

invalid_durations.head(10)

In [ ]:
df.groupby("Shift")["Daily Pay"].agg(
    ["count", "min", "mean", "median", "max", "std"]
).round(2)

In [ ]:
#calculate effective hourly pay 
df["Effective_Hourly_Pay"] = np.where(
    df["Hours_Worked"] > 0,
    df["Daily Pay"] / df["Hours_Worked"].replace(0, np.nan),
    np.nan
)

df[
    [
        "Employee",
        "Shift",
        "Hours_Worked",
        "Daily Pay",
        "Effective_Hourly_Pay"
    ]
].head(10)

In [ ]:
#compare effective hourly pay by type 
df.groupby("Shift")["Effective_Hourly_Pay"].agg(
    ["count", "min", "mean", "median", "max", "std"]
).round(2)

In [ ]:
#check  for payment inconsistencies 
# Working shifts with zero or negative daily pay
invalid_pay = df.loc[
    (df["Hours_Worked"] > 0)
    & (df["Daily Pay"] <= 0)
]

print("Working shifts with invalid pay:", len(invalid_pay))

# Rest days with non-zero pay
rest_pay = df.loc[
    (df["Shift"] == "Rest")
    & (df["Daily Pay"] != 0)
]

print("Rest days with non-zero pay:", len(rest_pay))

# Step 5: find differences by employee

In [ ]:
#Count the number of unique employees 
print("Unique employees:", df["Employee"].nunique())

print("\nShifts per employee:")
print(df["Employee"].value_counts().describe())

In [ ]:
# Calculate average hourly pay per employee and shift
employee_pay = df.pivot_table(
    index="Employee",
    columns="Shift",
    values="Effective_Hourly_Pay",
    aggfunc="mean"
)

# Keep employees with both day and night shifts
employee_pay = employee_pay.dropna(
    subset=["Day", "Night"]
).copy()

# Calculate the difference between the two
employee_pay["Pay_Difference"] = (
    employee_pay["Night"] - employee_pay["Day"]
)

employee_pay.head(10)


In [ ]:
#summarise the differences in pay between day abd night 
print(
    "Employees with both shift types:",
    len(employee_pay)
)

print(
    "Higher night-shift pay:",
    (employee_pay["Pay_Difference"] > 0).sum()
)

print(
    "Lower night-shift pay:",
    (employee_pay["Pay_Difference"] < 0).sum()
)

print(
    "Equal average pay:",
    (employee_pay["Pay_Difference"] == 0).sum()
)

print("\nPay difference summary:")
print(employee_pay["Pay_Difference"].describe())


In [ ]:
#visualise the differences between the two 
plt.figure(figsize=(9, 5))

plt.hist(
    employee_pay["Pay_Difference"],
    bins=20,
    edgecolor="black"
)

plt.axvline(
    x=0,
    color="red",
    linestyle="--",
    label="No difference"
)

plt.title("Night vs Day Effective Hourly Pay")
plt.xlabel("Night Pay - Day Pay (£/hour)")
plt.ylabel("Number of Employees")
plt.legend()
plt.show()


In [ ]:
# Find employee names appearing more than 31 times
df["Employee"].value_counts()[
    df["Employee"].value_counts() > 31
]

In [ ]:
# Examine the employee with the most records
employee_name = df["Employee"].value_counts().idxmax()

df.loc[
    df["Employee"] == employee_name,
    ["Record ID", "Employee", "Date", "Shift"]
].sort_values("Date").head(15)

# Step 6: validate employee identifiers

In [ ]:
# Create a provisional employee position identifier
df["Employee_Position"] = (
    (df["Record ID"] - 1) % 100
) + 1

# Check the two Michelle Reed records
df.loc[
    df["Employee"] == "Michelle Reed",
    [
        "Record ID",
        "Employee_Position",
        "Employee",
        "Date",
        "Shift"
    ]
].head(15)


In [ ]:
#check each employee maps to one name 
name_counts = df.groupby(
    "Employee_Position"
)["Employee"].nunique()

print(
    "Positions with multiple names:",
    (name_counts > 1).sum()
)

print(
    "Unique employee positions:",
    df["Employee_Position"].nunique()
)

print(
    "Records per position:"
)

print(
    df["Employee_Position"]
    .value_counts()
    .describe()
)


In [ ]:
# Check for duplicate employee-position/date combinations
duplicate_employee_dates = df.duplicated(
    subset=["Employee_Position", "Date"]
)

print(
    "Duplicate employee-date records:",
    duplicate_employee_dates.sum()
)


In [ ]:
# Create unique employee IDs
df["Employee_ID"] = (
    "EMP"
    + df["Employee_Position"].astype(str).str.zfill(3)
)

# Confirm the number of unique employees
print("Unique Employee IDs:", df["Employee_ID"].nunique())

# Check that each employee ID maps to one name
print(
    "IDs with multiple names:",
    (
        df.groupby("Employee_ID")["Employee"].nunique() > 1
    ).sum()
)

# Preview
df[["Employee_ID", "Employee", "Date", "Shift"]].head()


In [ ]:
from pathlib import Path

output_path = Path(r"C:\Users\themi\OneDrive\Documents\payroll project\ShiftsWorked_Cleaned.csv")

# Keep the original CSV unchanged
df.to_csv(output_path, index=False)

print("Saved:", output_path)

In [ ]:
#create employee dataset by employee_ID

# Select employee information
employees = df[
    ["Employee_ID", "Employee"]
].copy()

# Remove duplicate employee IDs
employees = employees.drop_duplicates(
    subset=["Employee_ID"]
)

# Rename columns to match PostgreSQL
employees = employees.rename(
    columns={
        "Employee_ID": "employee_id",
        "Employee": "employee_name"
    }
)

# Sort employees by ID
employees = employees.sort_values("employee_id")

# Preview
employees.head(10)


In [ ]:
# validate the employee dataset

print("Total employees:", len(employees))

print(
    "Duplicate employee IDs:",
    employees["employee_id"].duplicated().sum()
)

print(
    "Missing employee IDs:",
    employees["employee_id"].isna().sum()
)

print(
    "Missing employee names:",
    employees["employee_name"].isna().sum()
)

print(
    "IDs exceeding 6 characters:",
    (employees["employee_id"].str.len() > 6).sum()
)

print(
    "Names exceeding 100 characters:",
    (employees["employee_name"].str.len() > 100).sum()
)

In [ ]:
#export employees table to a csv file

from pathlib import Path

# Define export location
output_path = Path(r"C:\Users\themi\OneDrive\Documents\payroll project\employees.csv")

# Export without the Pandas index
employees.to_csv(output_path, index=False)

print("File saved to:", output_path.resolve())


In [ ]:
#prepare shifts csv file 

# Select the required columns
shifts = df[
    [
        "Record ID",
        "Employee_ID",
        "Date",
        "Shift",
        "Shift_Start_Datetime",
        "Shift_End_Datetime",
        "Hours_Worked",
        "Daily Pay"
    ]
].copy()

# Rename columns to match PostgreSQL
shifts.columns = [
    "record_id",
    "employee_id",
    "shift_date",
    "shift_type",
    "shift_start",
    "shift_end",
    "hours_worked",
    "daily_pay"
]

# Preview
shifts.head()


In [ ]:

print("Total shift records:", len(shifts))

print(
    "Duplicate record IDs:",
    shifts["record_id"].duplicated().sum()
)

print(
    "Missing employee IDs:",
    shifts["employee_id"].isna().sum()
)

print(
    "Invalid shift types:",
    (~shifts["shift_type"].isin(
        ["Day", "Night", "Rest"]
    )).sum()
)

print(
    "Negative working hours:",
    (shifts["hours_worked"] < 0).sum()
)

print(
    "Negative daily pay:",
    (shifts["daily_pay"] < 0).sum()
)

print(
    "Employee IDs not in employees table:",
    (~shifts["employee_id"].isin(
        employees["employee_id"]
    )).sum()
)


In [ ]:
#export to csv
from pathlib import Path

output_path = Path(r"C:\Users\themi\OneDrive\Documents\payroll project\shifts.csv")

shifts.to_csv(
    output_path,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

print("File saved to:", output_path.resolve())


In [ ]:

%pip install sqlalchemy psycopg2-binary


In [ ]:

import pandas as pd
from sqlalchemy import create_engine, text
from getpass import getpass
from urllib.parse import quote_plus


In [ ]:

# PostgreSQL connection details
username = "postgres"
host = "localhost"
port = "5432"
database = "workforce_payroll_db"

# Enter your PostgreSQL password securely
password = getpass("Enter PostgreSQL password: ")

# Create connection
engine = create_engine(
    f"postgresql+psycopg2://{username}:"
    f"{quote_plus(password)}@{host}:{port}/{database}"
)

# Test connection
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT current_database();")
    )
    print("Connected to:", result.scalar())


In [ ]:

query = """
SELECT
    rule_id,
    rule_name,
    rule_value,
    effective_from,
    effective_to
FROM pay_rules
ORDER BY rule_id;
"""

pay_rules = pd.read_sql(
    query,
    engine
)

pay_rules


In [ ]:

query = """
SELECT
    record_id,
    employee_id,
    shift_date,
    shift_type,
    shift_start,
    shift_end,
    hours_worked,
    daily_pay
FROM shifts
ORDER BY record_id;
"""

shift_data = pd.read_sql(
    query,
    engine
)

print("Total shift records:", len(shift_data))

shift_data.head()


In [ ]:

print("Shift records:", shift_data.shape)
print("Payroll rules:", pay_rules.shape)

print("\nShift data types:")
print(shift_data.dtypes)

print("\nMissing values:")
print(shift_data.isna().sum())


In [ ]:

from decimal import Decimal, ROUND_HALF_UP

# Retrieve the base hourly rate
base_rate = pay_rules.loc[
    pay_rules["rule_name"] == "base_hourly_rate",
    "rule_value"
].iloc[0]

# Convert to Decimal
base_rate = Decimal(str(base_rate))

print("Base hourly rate:", base_rate)
print("Data type:", type(base_rate))


In [ ]:

def calculate_ordinary_pay(hours, hourly_rate):
    """
    Calculate ordinary pay for a shift.

    Parameters:
        hours: Number of hours worked
        hourly_rate: Base hourly pay rate

    Returns:
        Ordinary pay rounded to 2 decimal places
    """

    hours = Decimal(str(hours))
    hourly_rate = Decimal(str(hourly_rate))

    pay = hours * hourly_rate

    return pay.quantize(
        Decimal("0.01"),
        rounding=ROUND_HALF_UP
    )


In [ ]:

print(
    "7-hour shift:",
    calculate_ordinary_pay(7, base_rate)
)

print(
    "8-hour shift:",
    calculate_ordinary_pay(8, base_rate)
)

print(
    "9-hour shift:",
    calculate_ordinary_pay(9, base_rate)
)

print(
    "Rest day:",
    calculate_ordinary_pay(0, base_rate)
)


In [ ]:

shift_data["Ordinary_Pay"] = shift_data[
    "hours_worked"
].apply(
    lambda hours: calculate_ordinary_pay(
        hours,
        base_rate
    )
)

# Preview results
shift_data[
    [
        "record_id",
        "employee_id",
        "shift_type",
        "hours_worked",
        "Ordinary_Pay"
    ]
].head(10)


In [ ]:

# Check missing calculated payments
print(
    "Missing ordinary pay:",
    shift_data["Ordinary_Pay"].isna().sum()
)

# Check negative payments
print(
    "Negative ordinary pay:",
    (shift_data["Ordinary_Pay"] < 0).sum()
)

# Check rest-day payments
rest_pay_check = shift_data.loc[
    shift_data["shift_type"] == "Rest",
    "Ordinary_Pay"
]

print(
    "Rest days with non-zero ordinary pay:",
    (rest_pay_check != 0).sum()
)


In [ ]:

# Convert payroll rules into a dictionary
rules = {
    row["rule_name"]: Decimal(str(row["rule_value"]))
    for _, row in pay_rules.iterrows()
}

print(rules)


In [ ]:
rules["night_multiplier"]

In [ ]:

from decimal import Decimal, ROUND_HALF_UP

def calculate_shift_pay(hours, shift_type, rules):
    """
    Calculate expected pay using fictional payroll rules.
    """

    hours = Decimal(str(hours))

    base_rate = rules["base_hourly_rate"]
    threshold = rules["overtime_threshold"]

    if hours < 0:
        raise ValueError("Hours worked cannot be negative")

    if shift_type == "Rest":
        if hours != 0:
            raise ValueError("Rest shifts must have zero hours")
        return Decimal("0.00")

    if shift_type not in ["Day", "Night"]:
        raise ValueError("Invalid shift type")

    # Separate ordinary and overtime hours
    ordinary_hours = min(hours, threshold)
    overtime_hours = max(
        hours - threshold,
        Decimal("0")
    )

    # Apply the appropriate shift multiplier
    if shift_type == "Day":
        multiplier = rules["day_multiplier"]
    else:
        multiplier = rules["night_multiplier"]

    # Calculate ordinary and overtime payments
    ordinary_pay = ordinary_hours * base_rate * multiplier

    overtime_pay = (
        overtime_hours
        * base_rate
        * rules["overtime_multiplier"]
    )

    total_pay = ordinary_pay + overtime_pay

    return total_pay.quantize(
        Decimal("0.01"),
        rounding=ROUND_HALF_UP
    )


In [ ]:

from decimal import Decimal, ROUND_HALF_UP

def calculate_shift_pay(hours, shift_type, rules):
    """
    Calculate expected pay using fictional payroll rules.
    """

    hours = Decimal(str(hours))

    base_rate = rules["base_hourly_rate"]
    threshold = rules["overtime_threshold"]

    if hours < 0:
        raise ValueError("Hours worked cannot be negative")

    if shift_type == "Rest":
        if hours != 0:
            raise ValueError("Rest shifts must have zero hours")
        return Decimal("0.00")

    if shift_type not in ["Day", "Night"]:
        raise ValueError("Invalid shift type")

    # Separate ordinary and overtime hours
    ordinary_hours = min(hours, threshold)
    overtime_hours = max(
        hours - threshold,
        Decimal("0")
    )

    # Apply the appropriate shift multiplier
    if shift_type == "Day":
        multiplier = rules["day_multiplier"]
    else:
        multiplier = rules["night_multiplier"]

    # Calculate ordinary and overtime payments
    ordinary_pay = ordinary_hours * base_rate * multiplier

    overtime_pay = (
        overtime_hours
        * base_rate
        * rules["overtime_multiplier"]
    )

    total_pay = ordinary_pay + overtime_pay

    return total_pay.quantize(
        Decimal("0.01"),
        rounding=ROUND_HALF_UP
    )


In [ ]:

test_cases = [
    (7, "Day", Decimal("140.00")),
    (7, "Night", Decimal("175.00")),
    (9, "Day", Decimal("190.00")),
    (9, "Night", Decimal("230.00")),
    (0, "Rest", Decimal("0.00"))
]

for hours, shift_type, expected in test_cases:
    actual = calculate_shift_pay(
        hours,
        shift_type,
        rules
    )

    print(
        f"{shift_type} ({hours} hours): "
        f"Expected = {expected}, Actual = {actual}"
    )

    assert actual == expected

print("All tests passed!")


In [ ]:

# Calculate expected pay for every shift
shift_data["Expected_Pay"] = shift_data.apply(
    lambda row: calculate_shift_pay(
        row["hours_worked"],
        row["shift_type"],
        rules
    ),
    axis=1
)

# Preview results
shift_data[
    [
        "record_id",
        "employee_id",
        "shift_type",
        "hours_worked",
        "daily_pay",
        "Expected_Pay"
    ]
].head(10)


In [ ]:

print(
    "Missing expected payments:",
    shift_data["Expected_Pay"].isna().sum()
)

print(
    "Negative expected payments:",
    (shift_data["Expected_Pay"] < 0).sum()
)

print(
    "Rest days with non-zero expected pay:",
    (
        shift_data.loc[
            shift_data["shift_type"] == "Rest",
            "Expected_Pay"
        ] != 0
    ).sum()
)

print(
    "Total expected payroll:",
    sum(shift_data["Expected_Pay"])
)


In [ ]:

from decimal import Decimal, ROUND_HALF_UP

# Convert recorded pay to Decimal
shift_data["Recorded_Pay_Decimal"] = (
    shift_data["daily_pay"]
    .apply(lambda x: Decimal(str(x)).quantize(
        Decimal("0.01"),
        rounding=ROUND_HALF_UP
    ))
)

# Calculate payment variance
shift_data["Pay_Variance"] = (
    shift_data["Recorded_Pay_Decimal"]
    - shift_data["Expected_Pay"]
)

# Preview
shift_data[
    [
        "record_id",
        "employee_id",
        "shift_type",
        "daily_pay",
        "Expected_Pay",
        "Pay_Variance"
    ]
].head(10)


In [ ]:

def classify_variance(variance):
    if variance > 0:
        return "Above Model"
    elif variance < 0:
        return "Below Model"
    else:
        return "Matches Model"

shift_data["Variance_Status"] = (
    shift_data["Pay_Variance"].apply(classify_variance)
)

shift_data["Variance_Status"].value_counts()


In [ ]:

# Calculate overall payroll reconciliation totals
total_recorded = sum(
    shift_data["Recorded_Pay_Decimal"]
)

total_expected = sum(
    shift_data["Expected_Pay"]
)

total_variance = sum(
    shift_data["Pay_Variance"]
)

print("Total recorded pay:", total_recorded)
print("Total modelled pay:", total_expected)
print("Net variance:", total_variance)

print("\nVariance classifications:")
print(shift_data["Variance_Status"].value_counts())


In [ ]:

variance_by_shift = shift_data.groupby(
    "shift_type"
).agg(
    Total_Records=("record_id", "count"),
    Recorded_Pay=("Recorded_Pay_Decimal", "sum"),
    Modelled_Pay=("Expected_Pay", "sum"),
    Net_Variance=("Pay_Variance", "sum")
)

variance_by_shift


In [ ]:

# Calculate absolute variance
shift_data["Absolute_Variance"] = (
    shift_data["Pay_Variance"].abs()
)

# Identify the 10 largest absolute variances
largest_variances = shift_data.sort_values(
    by="Absolute_Variance",
    ascending=False
).head(10)

# Display results
largest_variances[
    [
        "record_id",
        "employee_id",
        "shift_type",
        "hours_worked",
        "daily_pay",
        "Expected_Pay",
        "Pay_Variance"
    ]
]



In [ ]:

# Check that recorded pay equals modelled pay
# plus the calculated variance
reconciliation_errors = shift_data[
    shift_data["Recorded_Pay_Decimal"]
    != (
        shift_data["Expected_Pay"]
        + shift_data["Pay_Variance"]
    )
]

print(
    "Reconciliation errors:",
    len(reconciliation_errors)
)

# Check for missing variances
print(
    "Missing variances:",
    shift_data["Pay_Variance"].isna().sum()
)

# Verify overall reconciliation
assert total_recorded == total_expected + total_variance

print("Overall reconciliation passed!")


In [ ]:

shift_data.loc[
    shift_data["record_id"] == 10,
    [
        "record_id",
        "daily_pay",
        "Expected_Pay",
        "Pay_Variance"
    ]
]


In [ ]:

# Check mathematical consistency
reconciliation_errors = shift_data[
    shift_data["Recorded_Pay_Decimal"]
    != (
        shift_data["Expected_Pay"]
        + shift_data["Pay_Variance"]
    )
]

print("Reconciliation errors:", len(reconciliation_errors))

print(
    "Missing variances:",
    shift_data["Pay_Variance"].isna().sum()
)

# Verify overall totals
assert total_recorded == total_expected + total_variance

print("Overall reconciliation passed!")

# Check working shifts that exactly match the model
working_matches = shift_data.loc[
    (shift_data["shift_type"] != "Rest")
    & (shift_data["Pay_Variance"] == 0)
]

print("Working shifts matching model:", len(working_matches))


In [ ]:

# Create a separate DataFrame for database insertion
payroll_results_df = shift_data[
    [
        "record_id",
        "Recorded_Pay_Decimal",
        "Expected_Pay",
        "Pay_Variance",
        "Variance_Status"
    ]
].copy()

# Rename columns to match PostgreSQL
payroll_results_df = payroll_results_df.rename(
    columns={
        "Recorded_Pay_Decimal": "recorded_pay",
        "Expected_Pay": "modelled_pay",
        "Pay_Variance": "pay_variance",
        "Variance_Status": "variance_status"
    }
)

# Add rule-set version
payroll_results_df["rule_set_version"] = "v1_2015_01"

payroll_results_df.head()


In [ ]:

print("DataFrame shape:", payroll_results_df.shape)

print(
    "Missing values:",
    payroll_results_df.isna().sum()
)

print(
    "Duplicate record IDs:",
    payroll_results_df["record_id"].duplicated().sum()
)

print(
    "Rule-set versions:",
    payroll_results_df["rule_set_version"].unique()
)


In [ ]:

print(payroll_results_df.dtypes)

print(
    "\nRecorded pay type:",
    type(payroll_results_df["recorded_pay"].iloc[0])
)

print(
    "Modelled pay type:",
    type(payroll_results_df["modelled_pay"].iloc[0])
)

print(
    "Variance type:",
    type(payroll_results_df["pay_variance"].iloc[0])
)


In [ ]:

from sqlalchemy import text

# Insert calculated results into PostgreSQL
with engine.begin() as connection:
    payroll_results_df.to_sql(
        name="payroll_results",
        con=connection,
        if_exists="append",
        index=False,
        method="multi",
        chunksize=500
    )

print("Payroll results inserted successfully!")


In [ ]:
%pip install pytest

In [ ]:
import pytest

print("pytest version:", pytest.__version__)

In [ ]:

%%writefile payroll_engine.py

from decimal import Decimal, ROUND_HALF_UP


def calculate_shift_pay(hours, shift_type, rules):
    """
    Calculate modelled shift pay using fictional payroll rules.
    """

    hours = Decimal(str(hours))

    base_rate = rules["base_hourly_rate"]
    threshold = rules["overtime_threshold"]

    if hours < 0:
        raise ValueError("Hours worked cannot be negative")

    if shift_type == "Rest":
        if hours != 0:
            raise ValueError("Rest shifts must have zero hours")
        return Decimal("0.00")

    if shift_type not in ["Day", "Night"]:
        raise ValueError("Invalid shift type")

    ordinary_hours = min(hours, threshold)
    overtime_hours = max(
        hours - threshold,
        Decimal("0")
    )

    if shift_type == "Day":
        multiplier = rules["day_multiplier"]
    else:
        multiplier = rules["night_multiplier"]

    ordinary_pay = (
        ordinary_hours
        * base_rate
        * multiplier
    )

    overtime_pay = (
        overtime_hours
        * base_rate
        * rules["overtime_multiplier"]
    )

    total_pay = ordinary_pay + overtime_pay

    return total_pay.quantize(
        Decimal("0.01"),
        rounding=ROUND_HALF_UP
    )


In [ ]:

from payroll_engine import calculate_shift_pay

result = calculate_shift_pay(
    9,
    "Night",
    rules
)

print("9-hour night shift:", result)


In [ ]:

import os

print("Current working directory:", os.getcwd())

print(
    "Payroll engine file exists:",
    os.path.exists("payroll_engine.py")
)


In [ ]:

%%writefile test_payroll_engine.py

import pytest
from decimal import Decimal

from payroll_engine import calculate_shift_pay


# Fictional payroll rules used for testing
TEST_RULES = {
    "base_hourly_rate": Decimal("20.00"),
    "day_multiplier": Decimal("1.00"),
    "night_multiplier": Decimal("1.25"),
    "overtime_multiplier": Decimal("1.50"),
    "overtime_threshold": Decimal("8.00")
}


def test_day_shift():
    result = calculate_shift_pay(
        7, "Day", TEST_RULES
    )
    assert result == Decimal("140.00")


def test_night_shift():
    result = calculate_shift_pay(
        7, "Night", TEST_RULES
    )
    assert result == Decimal("175.00")


def test_day_overtime():
    result = calculate_shift_pay(
        9, "Day", TEST_RULES
    )
    assert result == Decimal("190.00")


def test_night_overtime():
    result = calculate_shift_pay(
        9, "Night", TEST_RULES
    )
    assert result == Decimal("230.00")


def test_rest_shift():
    result = calculate_shift_pay(
        0, "Rest", TEST_RULES
    )
    assert result == Decimal("0.00")


In [ ]:
def test_night_shift():
    result = calculate_shift_pay(
        7, "Night", TEST_RULES
    )
    assert result == Decimal("175.00")

In [ ]:

import pytest

print("pytest version:", pytest.__version__)

# Run automated tests
exit_code = pytest.main([
    "test_payroll_engine.py",
    "-v"
])

print("pytest exit code:", exit_code)


In [ ]:

%%writefile -a test_payroll_engine.py


@pytest.mark.parametrize(
    "hours, shift_type, expected",
    [
        (7.5, "Day", "150.00"),
        (8, "Day", "160.00"),
        (8.5, "Day", "175.00"),
        (8, "Night", "200.00"),
        (8.5, "Night", "215.00")
    ]
)
def test_overtime_boundaries(hours, shift_type, expected):
    result = calculate_shift_pay(
        hours,
        shift_type,
        TEST_RULES
    )

    assert result == Decimal(expected)


In [ ]:

%%writefile -a test_payroll_engine.py


def test_negative_hours():
    with pytest.raises(ValueError):
        calculate_shift_pay(
            -2,
            "Day",
            TEST_RULES
        )


def test_invalid_shift_type():
    with pytest.raises(ValueError):
        calculate_shift_pay(
            8,
            "Evening",
            TEST_RULES
        )


def test_rest_shift_with_hours():
    with pytest.raises(ValueError):
        calculate_shift_pay(
            4,
            "Rest",
            TEST_RULES
        )


In [ ]:
if hours < 0:
    raise ValueError("Hours worked cannot be negative")

In [ ]:

import pytest

exit_code = pytest.main([
    "test_payroll_engine.py",
    "-v"
])

print("pytest exit code:", exit_code)


In [ ]:

import sys
import subprocess

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "test_payroll_engine.py",
        "-v"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)


In [ ]:

from pathlib import Path

test_file = Path("test_payroll_engine.py")

print("File exists:", test_file.exists())
print("File location:", test_file.resolve())

print("\nLast 45 lines:")
print("\n".join(test_file.read_text().splitlines()[-45:]))


In [ ]:

%%writefile -a test_payroll_engine.py


@pytest.mark.parametrize(
    "hours, expected",
    [
        ("1", "20.13"),
        ("2", "40.25"),
        ("3", "60.38")
    ]
)
def test_decimal_rounding(hours, expected):

    custom_rules = TEST_RULES.copy()

    custom_rules["base_hourly_rate"] = Decimal("20.125")

    result = calculate_shift_pay(
        hours,
        "Day",
        custom_rules
    )

    assert result == Decimal(expected)


In [ ]:

import sys
import subprocess

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "test_payroll_engine.py",
        "-v"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)


In [ ]:

%%writefile -a payroll_engine.py


def calculate_pay_variance(recorded_pay, modelled_pay):
    """
    Calculate the difference between recorded
    and modelled payroll amounts.
    """

    recorded_pay = Decimal(str(recorded_pay))
    modelled_pay = Decimal(str(modelled_pay))

    variance = recorded_pay - modelled_pay

    return variance.quantize(
        Decimal("0.01"),
        rounding=ROUND_HALF_UP
    )


In [ ]:

%%writefile -a test_payroll_engine.py


from payroll_engine import calculate_pay_variance


@pytest.mark.parametrize(
    "recorded, modelled, expected",
    [
        ("78.06", "175.00", "-96.94"),
        ("202.98", "140.00", "62.98"),
        ("0.00", "0.00", "0.00"),
        ("160.00", "160.00", "0.00")
    ]
)
def test_pay_variance(recorded, modelled, expected):

    result = calculate_pay_variance(
        recorded,
        modelled
    )

    assert result == Decimal(expected)

    # Verify the reconciliation equation
    assert (
        Decimal(recorded)
        == Decimal(modelled) + result
    )


In [ ]:

import sys
import subprocess

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "test_payroll_engine.py",
        "-v"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)


In [ ]:

from pathlib import Path

project_path = Path.home() / "Documents" / "workforce-payroll-analytics"

folders = [
    "data",
    "notebooks",
    "src",
    "sql",
    "tests",
    "powerbi",
    "docs"
]

project_path.mkdir(parents=True, exist_ok=True)

for folder in folders:
    (project_path / folder).mkdir(exist_ok=True)

print("Project directory:", project_path)

for folder in folders:
    print("Created:", folder)


In [ ]:

print("Project exists:", project_path.exists())

for folder in folders:
    folder_path = project_path / folder

    print(
        folder,
        "exists:",
        folder_path.is_dir()
    )


In [ ]:

from pathlib import Path
import shutil

source_path = Path.home()

# Copy payroll calculation module
shutil.copy2(
    source_path / "payroll_engine.py",
    project_path / "src" / "payroll_engine.py"
)

# Copy automated tests
shutil.copy2(
    source_path / "test_payroll_engine.py",
    project_path / "tests" / "test_payroll_engine.py"
)

print("Python files copied successfully!")


In [ ]:

payroll_file = project_path / "src" / "payroll_engine.py"

test_file = project_path / "tests" / "test_payroll_engine.py"

print("Payroll engine exists:", payroll_file.exists())
print("Test file exists:", test_file.exists())

print("\nProject files:")

for file in project_path.rglob("*.py"):
    print(file.relative_to(project_path))


In [ ]:

from pathlib import Path

test_file = project_path / "tests" / "test_payroll_engine.py"

# Read the existing test file
test_code = test_file.read_text()

# Update the imports
test_code = test_code.replace(
    "from payroll_engine import calculate_shift_pay",
    "from src.payroll_engine import calculate_shift_pay"
)

test_code = test_code.replace(
    "from payroll_engine import calculate_pay_variance",
    "from src.payroll_engine import calculate_pay_variance"
)

# Save the updated file
test_file.write_text(test_code)

print("Test imports updated successfully!")


In [ ]:

test_code = test_file.read_text()

for line in test_code.splitlines():
    if line.startswith("from "):
        print(line)


In [ ]:

import sys
import subprocess

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_payroll_engine.py",
        "-v"
    ],
    cwd=str(project_path),
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)


In [ ]:

from pathlib import Path

requirements = """pandas
SQLAlchemy
psycopg2-binary
pytest
notebook
"""

requirements_file = project_path / "requirements.txt"

requirements_file.write_text(requirements)

print("Requirements file created!")
print("Location:", requirements_file)


In [ ]:

print(requirements_file.read_text())


In [ ]:


sql_file = project_path / "sql" / "01_create_tables.sql"

print("SQL file exists:", sql_file.exists())

if sql_file.exists():
    print("File size:", sql_file.stat().st_size, "bytes")
    print("File location:", sql_file)


In [ ]:

from pathlib import Path

sql_folder = project_path / "sql"

print("Expected folder:", sql_folder)
print("Folder exists:", sql_folder.exists())

print("\nFiles currently in SQL folder:")

for file in sql_folder.iterdir():
    print(file.name)


In [ ]:

sql_file = project_path / "sql" / "01_create_tables.sql"

print("SQL file exists:", sql_file.is_file())
print("File size:", sql_file.stat().st_size, "bytes")

print("\nFirst 10 lines:")
print("\n".join(sql_file.read_text(encoding="utf-8-sig").splitlines()[:10]))



In [ ]:

rules_file = project_path / "sql" / "02_insert_pay_rules.sql"

print("File exists:", rules_file.is_file())

if rules_file.is_file():
    print("File size:", rules_file.stat().st_size, "bytes")
    print("\nFirst 10 lines:")
    print("\n".join(
        rules_file.read_text(encoding="utf-8-sig").splitlines()[:10]
    ))

In [ ]:

sql_folder = project_path / "sql"

print("SQL folder:", sql_folder)
print("\nFiles currently saved:")

for file in sql_folder.iterdir():
    print(file.name)


In [ ]:

rules_file = project_path / "sql" / "02_insert_pay_rules.sql"

print("File exists:", rules_file.is_file())

if rules_file.is_file():
    print("File size:", rules_file.stat().st_size, "bytes")


In [ ]:

rules_file = project_path / "sql" / "02_insert_pay_rules.sql"

print("File exists:", rules_file.is_file())

if rules_file.is_file():
    print("File size:", rules_file.stat().st_size, "bytes")

    content = rules_file.read_text(encoding="utf-8-sig")

    print("Contains INSERT statement:", "INSERT INTO pay_rules" in content)
    print("Contains five rule names:", all(
        rule in content
        for rule in [
            "base_hourly_rate",
            "day_multiplier",
            "night_multiplier",
            "overtime_multiplier",
            "overtime_threshold"
        ]
    ))


In [ ]:

views_file = project_path / "sql" / "03_create_views.sql"

print("File exists:", views_file.is_file())

if views_file.is_file():

    content = views_file.read_text(encoding="utf-8-sig")

    print("File size:", views_file.stat().st_size, "bytes")

    print(
        "Detail view found:",
        "vw_payroll_detail" in content
    )

    print(
        "Employee summary view found:",
        "vw_employee_payroll_summary" in content
    )

    print(
        "Contains CREATE VIEW statements:",
        content.count("CREATE OR REPLACE VIEW") == 2
    )


In [ ]:

validation_file = project_path / "sql" / "04_data_validation.sql"

print("File exists:", validation_file.is_file())

if validation_file.is_file():
    content = validation_file.read_text(encoding="utf-8-sig")

    print("File size:", validation_file.stat().st_size, "bytes")
    print("SELECT statements:", content.count("SELECT"))
    print("Reconciliation check found:", "reconciliation_errors" in content)
    print("Variance status check found:", "incorrect_variance_status" in content)


In [ ]:

from pathlib import Path

# Check the current Jupyter working directory
current_folder = Path.cwd()

print("Current folder:", current_folder)

# Look for the two exported datasets
for filename in ["employees.csv", "shifts.csv"]:

    file_path = current_folder / filename

    print(
        filename,
        "exists:",
        file_path.is_file()
    )

    if file_path.is_file():
        print("Location:", file_path)


In [ ]:

from pathlib import Path

search_folders = [
    Path.home() / "Documents",
    Path.home() / "Downloads",
    Path.home() / "Desktop",
    Path.home() / "OneDrive" / "Documents"
]

target_files = {"employees.csv", "shifts.csv"}

found_files = []

for folder in search_folders:
    if folder.exists():
        print("Searching:", folder)

        for file in folder.rglob("*.csv"):
            if file.name.lower() in target_files:
                found_files.append(file)
                print("FOUND:", file)

print("\nTotal matching files:", len(found_files))


In [ ]:

from pathlib import Path
import shutil

source_folder = (
    Path.home()
    / "OneDrive"
    / "Documents"
    / "payroll project"
)

destination_folder = project_path / "data"

for filename in ["employees.csv", "shifts.csv"]:

    source_file = source_folder / filename
    destination_file = destination_folder / filename

    shutil.copy2(source_file, destination_file)

    print("Copied:", filename)

print("\nDestination:", destination_folder)


In [ ]:

import pandas as pd

employees_check = pd.read_csv(
    project_path / "data" / "employees.csv"
)

shifts_check = pd.read_csv(
    project_path / "data" / "shifts.csv"
)

print("Employee records:", len(employees_check))
print("Shift records:", len(shifts_check))

print("\nEmployee columns:")
print(employees_check.columns.tolist())

print("\nShift columns:")
print(shifts_check.columns.tolist())

print("\nMissing Employee IDs:", employees_check["employee_id"].isna().sum())
print("Missing Shift Employee IDs:", shifts_check["employee_id"].isna().sum())


In [ ]:

from pathlib import Path

source_folder = (
    Path.home()
    / "OneDrive"
    / "Documents"
    / "payroll project"
)

print("Jupyter notebooks found:\n")

notebooks = list(source_folder.rglob("*.ipynb"))

for notebook in notebooks:
    if ".ipynb_checkpoints" not in notebook.parts:
        print(notebook)


In [ ]:

from pathlib import Path

search_folders = [
    Path.home(),
    Path.home() / "Documents",
    Path.home() / "OneDrive" / "Documents"
]

found_notebooks = set()

for folder in search_folders:
    if folder.exists():
        for notebook in folder.rglob("*.ipynb"):
            if ".ipynb_checkpoints" not in notebook.parts:
                found_notebooks.add(notebook)

print("Notebooks found:", len(found_notebooks))

for notebook in sorted(found_notebooks):
    print(notebook)


In [ ]:

# Keywords related to our payroll project
keywords = [
    "payroll",
    "workforce",
    "shift",
    "employee"
]

matching_notebooks = [
    notebook
    for notebook in found_notebooks
    if any(
        keyword in notebook.name.lower()
        or keyword in str(notebook.parent).lower()
        for keyword in keywords
    )
    and "anaconda3" not in str(notebook).lower()
]

print("Matching notebooks:", len(matching_notebooks))

for notebook in sorted(matching_notebooks):
    print(notebook)


In [ ]:

from pathlib import Path
import shutil

source_notebook = Path.home() / "payroll_project.ipynb"

destination_notebook = (
    project_path
    / "notebooks"
    / "payroll_project.ipynb"
)

shutil.copy2(
    source_notebook,
    destination_notebook
)

print("Notebook copied successfully!")
print("Destination:", destination_notebook)
print("File exists:", destination_notebook.is_file())


In [ ]:

import json

with open(destination_notebook, "r", encoding="utf-8") as file:
    notebook_data = json.load(file)

cells = notebook_data.get("cells", [])

code_cells = [
    cell for cell in cells
    if cell["cell_type"] == "code"
]

markdown_cells = [
    cell for cell in cells
    if cell["cell_type"] == "markdown"
]

print("Total cells:", len(cells))
print("Code cells:", len(code_cells))
print("Markdown cells:", len(markdown_cells))


In [ ]:

import re

patterns = {
    "Password reference": r"password|passwd|pwd",
    "Database connection": r"postgresql://|create_engine|psycopg2",
    "Potential secret": r"api[_-]?key|secret|token",
    "Local Windows path": r"C:\\Users\\"
}

for name, pattern in patterns.items():

    matching_cells = []

    for index, cell in enumerate(cells, start=1):

        source = "".join(cell.get("source", []))

        if re.search(pattern, source, re.IGNORECASE):
            matching_cells.append(index)

    print(f"{name}: {len(matching_cells)} cells")
    print("Cell numbers:", matching_cells)
    print()


In [ ]:

import re

# Cell 42 (Python lists start counting at zero)
cell_42 = cells[41]

source = "".join(cell_42.get("source", []))

print("Cell 42 security checks:\n")

print(
    "Uses getpass:",
    bool(re.search(r"\bgetpass\s*\(", source))
)

print(
    "Contains create_engine:",
    "create_engine" in source
)

print(
    "Contains a PostgreSQL connection:",
    "postgresql" in source.lower()
)

print(
    "Contains password assignment:",
    bool(re.search(r"\bpassword\s*=", source))
)

print(
    "Contains saved outputs:",
    bool(cell_42.get("outputs", []))
)


In [ ]:

import re

cell_42 = cells[41]
source = "".join(cell_42.get("source", []))

# Identify password assignment without printing its value
assignment = re.search(
    r"\bpassword\s*=\s*(.+)",
    source
)

if assignment:
    expression = assignment.group(1).strip()

    if re.match(r"getpass\s*\(", expression):
        print("Password assignment: Uses getpass")
    elif re.match(r"['\"]", expression):
        print("Password assignment: WARNING - possible hard-coded value")
    else:
        print("Password assignment: Requires further review")
else:
    print("No direct password assignment identified")

# Inspect saved output types without displaying contents
outputs = cell_42.get("outputs", [])

print("\nSaved output count:", len(outputs))

for index, output in enumerate(outputs, start=1):
    print(
        "Output", index,
        "- Type:", output.get("output_type")
    )


In [ ]:

import json

# Load the copied notebook
with open(destination_notebook, "r", encoding="utf-8") as file:
    notebook_data = json.load(file)

cleared_cells = 0

for cell in notebook_data["cells"]:

    if cell["cell_type"] == "code":

        if cell.get("outputs"):
            cleared_cells += 1

        # Remove saved outputs
        cell["outputs"] = []

        # Reset execution counter
        cell["execution_count"] = None

# Save the cleaned copy
with open(destination_notebook, "w", encoding="utf-8") as file:
    json.dump(notebook_data, file, indent=1, ensure_ascii=False)

print("Notebook outputs cleared successfully!")
print("Cells with outputs removed:", cleared_cells)
print("Original notebook remains unchanged.")


In [ ]:

with open(destination_notebook, "r", encoding="utf-8") as file:
    cleaned_notebook = json.load(file)

cells = cleaned_notebook["cells"]

remaining_outputs = sum(
    len(cell.get("outputs", []))
    for cell in cells
    if cell["cell_type"] == "code"
)

code_count = sum(
    cell["cell_type"] == "code"
    for cell in cells
)

markdown_count = sum(
    cell["cell_type"] == "markdown"
    for cell in cells
)

print("Total cells:", len(cells))
print("Code cells:", code_count)
print("Markdown cells:", markdown_count)
print("Remaining saved outputs:", remaining_outputs)


In [ ]:

import re

path_cells = [3, 33, 36, 39]

for cell_number in path_cells:
    cell = cells[cell_number - 1]
    source = "".join(cell.get("source", []))

    print(f"\nCell {cell_number}")

    print(
        "Reads CSV:",
        "read_csv" in source
    )

    print(
        "Writes CSV:",
        "to_csv" in source
    )

    print(
        "Uses Path:",
        bool(re.search(r"\bPath\s*\(", source))
    )

    print(
        "Uses absolute Windows path:",
        bool(re.search(r"C:\\Users\\", source, re.IGNORECASE))
    )


In [ ]:

import re
from pathlib import PureWindowsPath

for cell_number in [3, 33, 36, 39]:

    source = "".join(
        cells[cell_number - 1].get("source", [])
    )

    filenames = re.findall(
        r"[\w .()\-]+\.csv",
        source,
        flags=re.IGNORECASE
    )

    filenames = [
        PureWindowsPath(name.strip()).name
        for name in filenames
    ]

    print(f"Cell {cell_number}:")
    print("CSV filenames:", sorted(set(filenames)))
    print()


In [ ]:

from pathlib import Path

data_folder = project_path / "data"

required_files = [
    "ShiftsWorked.csv",
    "ShiftsWorked_Cleaned.csv",
    "employees.csv",
    "shifts.csv"
]

for filename in required_files:
    file_path = data_folder / filename

    print(
        filename,
        "->",
        "FOUND" if file_path.is_file() else "MISSING"
    )


In [ ]:

import re
from pathlib import Path

cell_3_source = "".join(cells[2].get("source", []))

# Find the original CSV path without printing it
matches = re.findall(
    r'[A-Za-z]:\\[^"\']*ShiftsWorked\.csv',
    cell_3_source,
    flags=re.IGNORECASE
)

if matches:
    original_path = Path(matches[0])

    print("Original CSV path identified: Yes")
    print("File exists:", original_path.is_file())
    print("Filename:", original_path.name)
else:
    print("Original CSV path identified: No")


In [ ]:

import shutil

destination_csv = project_path / "data" / "ShiftsWorked.csv"

shutil.copy2(original_path, destination_csv)

print("Original dataset copied successfully!")
print("File exists:", destination_csv.is_file())
print("Filename:", destination_csv.name)


In [ ]:

import pandas as pd

raw_df = pd.read_csv(destination_csv)

print("Number of rows:", len(raw_df))
print("Number of columns:", len(raw_df.columns))
print("Column names:", raw_df.columns.tolist())


In [ ]:

from pathlib import Path

# The notebook lives inside the notebooks folder.
# Its parent folder is the project root.
project_root = destination_notebook.parent.parent

data_dir = project_root / "data"

raw_csv = data_dir / "ShiftsWorked.csv"
cleaned_csv = data_dir / "ShiftsWorked_Cleaned.csv"
employees_csv = data_dir / "employees.csv"
shifts_csv = data_dir / "shifts.csv"

print("Project folder identified:", project_root.is_dir())
print("Data folder identified:", data_dir.is_dir())
print("Original dataset found:", raw_csv.is_file())
print("Employees dataset found:", employees_csv.is_file())
print("Shifts dataset found:", shifts_csv.is_file())
print("Intermediate cleaned dataset exists:", cleaned_csv.is_file())


In [ ]:

from pathlib import Path

# Find the project root from the current working directory
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

data_dir = project_root / "data"

print("Project root:", project_root)
print("Data folder exists:", data_dir.is_dir())


In [ ]:

from pathlib import Path

# Locate the payroll project inside Documents
project_root = Path.home() / "Documents" / "workforce-payroll-analytics"

# Define the data folder
data_dir = project_root / "data"

print("Project root:", project_root)
print("Project folder exists:", project_root.is_dir())
print("Data folder exists:", data_dir.is_dir())
print("Original CSV exists:", (data_dir / "ShiftsWorked.csv").is_file())


In [ ]:

print("Rows:", len(shifts_df))
print("Columns:", len(shifts_df.columns))
